# Tutorial: inflating ensembles 

While ensembles are useful data format to store nested histograms in, they are somewhat unwieldy and require many special functions to handle. The largest problem is, however, that they do not work well with vector-based operations, and require recursive traversal by the convolution code. 

To combat this we can `inflate` ensembles by changing their nested structure to a flattened format. 

In [1]:
import json
import pkg_resources
from syntheticstellarpopconvolve.ensemble_utils import convert_ensemble_to_dataframe

# load the data
example_ensemble_filename = pkg_resources.resource_filename(
    "syntheticstellarpopconvolve", "example_data/example_ensemble.json"
)
with open(example_ensemble_filename, "r") as f_ensemble:
    ensemble = json.loads(f_ensemble.read())

We can then inflate this ensmeble by using the `convert_ensemble_to_dataframe` function.

It is best to already know the structure of the ensemble, so you know exactly which subtree you want to take, and whether it contains named layers. If the ensemble contains named layers, the structure should be `named_layer_1, value_layer_1, ... named_layer_n, value_layer_n, normalized_yield_layer_n`.

```python
inflate_ensemble = convert_ensemble_to_dataframe(
    ensemble_data, # subtree of ensemble
    verbose=False, # flag to show info while inflating
    contains_named_layers=True, # flag to indicate whether the ensemble contains named layer (i.e. those that indicate what is in the next layer) 
)
```

Particularly, if you indicate that the ensemble contains named layers, the first layer _should_ be a named layer. If it does not, or somehow the structure is not like it should be, the read-out is misaligned. If that is the case, please double check you provided the correct subtree, or wrap it in {'ensemble': ensemble_data}

In [15]:
inflated_ensemble = convert_ensemble_to_dataframe(
    ensemble_data=ensemble["ensemble"]['Xyield'],
    verbose=False,
    contains_named_layers=True,
)

print(inflated_ensemble.head())

['time', 'source', 'isotope']
   time source isotope probability
0  -0.1   Wind    Al27         0.0
1  -0.1   Wind    Ar36         0.0
2  -0.1   Wind    Ar38         0.0
3  -0.1   Wind    Ar40         0.0
4  -0.1   Wind     B10         0.0


The inflated ensemble by default calls the final layer 'probability', but that data can ofcourse be anything depending on the pop-synth simulation output.

Moreover, the object-type of all columns are by default string based (except for the final layer). This is because, when reading out the ensemble, we do not want to impose any type on the data, and everything can be converted to a string, but not everything can be converted to a numerical type.

In [16]:
print(inflated_ensemble.dtypes)

time           object
source         object
isotope        object
probability    object
dtype: object


After inflating the ensemble you should convert the columns to their actual types

In [20]:
inflated_ensemble = inflated_ensemble.astype({'time': 'float'})
print(inflated_ensemble.dtypes)

time           float64
source          object
isotope         object
probability     object
dtype: object


In [21]:
print(10**inflated_ensemble['time'])

0             0.794328
1             0.794328
2             0.794328
3             0.794328
4             0.794328
              ...     
102787    15848.931925
102788    15848.931925
102789    15848.931925
102790    15848.931925
102791    15848.931925
Name: time, Length: 102792, dtype: float64


lets get the unique elements instead of the isotopes:

In [3]:
unique_isotopes = inflated_ensemble['isotope'].unique()

In [4]:
print(unique_isotopes)

['Al27' 'Ar36' 'Ar38' 'Ar40' 'B10' 'B11' 'Be9' 'C12' 'C13' 'Ca40' 'Ca42'
 'Ca43' 'Ca44' 'Ca46' 'Ca48' 'Cl35' 'Cl37' 'Co59' 'Cr50' 'Cr52' 'Cr53'
 'Cr54' 'Cu63' 'Cu65' 'F19' 'Fe54' 'Fe56' 'Fe57' 'Fe58' 'Ga69' 'Ga71'
 'Ge70' 'H1' 'He3' 'He4' 'K39' 'K40' 'K41' 'Li6' 'Li7' 'Mg24' 'Mg25'
 'Mg26' 'Mn55' 'N14' 'N15' 'Na23' 'Ne20' 'Ne21' 'Ne22' 'Ni58' 'Ni60'
 'Ni62' 'Ni64' 'O16' 'O17' 'O18' 'P31' 'Ru99' 'S32' 'S33' 'S34' 'S36'
 'Sc45' 'Si28' 'Si29' 'Si30' 'Ti46' 'Ti47' 'Ti48' 'Ti49' 'Ti50' 'V50'
 'V51' 'Zn64' 'Zn66' 'Zn67' 'Zn68' 'Zn70' 'Ag107' 'Ag109' 'Al26' 'As75'
 'Au197' 'Ba134' 'Ba135' 'Ba136' 'Ba137' 'Ba138' 'Bi209' 'Br79' 'Br81'
 'Cd108' 'Cd110' 'Cd111' 'Cd112' 'Cd113' 'Cd114' 'Cd116' 'Ce140' 'Ce142'
 'Cs133' 'Dy160' 'Dy161' 'Dy162' 'Dy163' 'Dy164' 'Er164' 'Er166' 'Er167'
 'Er168' 'Er170' 'Eu151' 'Eu153' 'Gd152' 'Gd154' 'Gd155' 'Gd156' 'Gd157'
 'Gd158' 'Gd160' 'Ge72' 'Ge73' 'Ge74' 'Ge76' 'Hf176' 'Hf177' 'Hf178'
 'Hf179' 'Hf180' 'Hg198' 'Hg199' 'Hg200' 'Hg201' 'Hg202' 'Hg204' 'Ho165'
 'I1

In [9]:
import numpy as np
# remove numbers
cleaned_isotopes = [''.join([i for i in isotope if not i.isdigit()]) for isotope in unique_isotopes] 
unique_elements = np.unique(np.array(cleaned_isotopes))

array(['Ag', 'Al', 'Ar', 'As', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Br', 'C',
       'Ca', 'Cd', 'Ce', 'Cl', 'Co', 'Cr', 'Cs', 'Cu', 'Dy', 'Er', 'Eu',
       'F', 'Fe', 'Ga', 'Gd', 'Ge', 'H', 'He', 'Hf', 'Hg', 'Ho', 'I',
       'In', 'Ir', 'K', 'Kr', 'La', 'Li', 'Lu', 'Mg', 'Mn', 'Mo', 'N',
       'Na', 'Nb', 'Nd', 'Ne', 'Ni', 'O', 'Os', 'P', 'Pb', 'Pd', 'Pm',
       'Po', 'Pr', 'Pt', 'Rb', 'Re', 'Rh', 'Ru', 'S', 'Sb', 'Sc', 'Se',
       'Si', 'Sm', 'Sn', 'Sr', 'Ta', 'Tb', 'Tc', 'Te', 'Ti', 'Tl', 'Tm',
       'V', 'W', 'Xe', 'Y', 'Yb', 'Zn', 'Zr'], dtype='<U2')